
# Documentación de Decisiones Técnicas: Pipeline Inmobiliario

- **Proyecto:** Procesamiento de Datos de Propiedades (ETL)
- **Arquitectura:** Medallion (Bronze, Silver, Gold)
- **Modelo de Datos:** Dimensional (Star Schema)
- **Orquestacion** Databricks Worflows
- **Tecnología:** Databricks (SQL )

---
## Estrategia de Arquitectura: Medallion
Se ha optado por una arquitectura de tres capas para garantizar la trazabilidad y la calidad del dato.

### Capa Bronze (Raw Data)
**Propósito:** Almacenamiento preservando la informacion

**Decisión:** La ingesta hacia **Bronze** es **append-only** desde un **archivo csv**.Sin ningun tipo de transformaciones

**Formato:** Delta Lake (para soporte de **transacciones ACID**).

### Capa Silver (Cleaning)
**Propósito:** Datos limpios, normalizados y deduplicados.

**Decisiones Técnicas:**

**Limpieza::** de espacios vacios, tipo de datos adecuados y texto a minuscula

**Filtrado:** valores nulos en columna como precio, rangos validos en ambientes(1-6), imputar valores placeholder(999) en columna como antiguedad

**Normalización:** 
- Estandarización de zonas (GBA Norte, Sur, Oeste) mediante lógica de REGEXP.
- Estandarizacion de estado (excelente,bueno,regular, a_estrenar,a_refaccionar, en_desarrollo) mediante lógica de RLIKE
- Estandarizacion de tipo_operacion (alquiler, venta, venta_alquiler) mediante lógica de RLIKE
- Derivacion de columna, mediante logica REGEXP, se creo una nueva columna (ciudad)

**Deduplicación:** Uso de ROW_NUMBER() sobre url para asegurar unicidad.

**Soporte:** Operaciones MERGE UPSERT(la url se define como clave natural), para gestionar la integración de datos y garantizar procesos idempotente.

### Capa Gold 
Propósito: Responder preguntas de negocio puntuales

Decisión: Implementación de un Modelo Dimensional (Star Schema) y KPIs aregados para el analisis.

**Modelo Dimensional:** 
**Star Schema** Para optimizar las consultas analíticas, los datos de la capa Silver se transforman en una estructura de Hechos y Dimensiones.

**Tabla de Hechos: fact_propiedades**
Contiene las métricas cuantitativas y las claves foráneas:

- metricas: precio, ambientes, expensas, metros_cuadrados_totales, precio_por_m2.
- Claves foraneas: zona_id, estado_id, tipo_operacion_id, id_fecha

**Tablas de Dimensiones:**

- dim_zona: Atributos geográficos (Ciudad, Provincia, zona). Con soporte **SCD Type 2**
- dim_tipo_operacion: Categorías (Venta, Alquiler).
- dim_estado: Categorización del estado de la propiedad (excelente,bueno,regular, a_estrenar,a_refaccionar, en_desarrollo ).
- dim_fecha: Rango de 10 años partiendo (5 atrás, 5 adelante)
---
### 📋 Hallazgos del EDA

### Problemas de calidad identificados:

1. **Valores placeholder:** El valor `999` en antigüedad representa datos faltantes
2. **Campos nulos:** Varios campos como `expensas`, `orientacion`, `cochera` tienen alto % de nulos
3. **Datos mixtos:** Precios en USD y ARS mezclados (requiere normalización)
4. **Posibles duplicados:** Propiedades repetidas con misma ubicación y precio
5. **Outliers:** Precios extremadamente altos o bajos que podrían ser errores

### Transformaciones necesarias para capa Silver:

- [ ] Convertir NULL en antigüedad a `999`
- [ ] Normalizar precios a una sola moneda (o crear columnas separadas)
- [ ] Eliminar duplicados
- [ ] Filtrar outliers extremos
- [ ] Limpiar espacio vacios
- [ ] Normalizar texto a minusculas
- [ ] Estandarizar columnas como `zona`, `estado`, `tipo_de_operacion`
- [ ] Derivacion de columnas, para crear una nueva columna (ciudad)
- [ ] Calcular métricas derivadas (precio por m2)
